# Build sum-of-products KEO for $\mathrm{CH_3F}$ using user-defined coordinates

In [1]:
import jax
import numpy as np
from jax import numpy as jnp
from scipy import constants

from keosop.gmat import Gmat
from keosop.hdf5 import save_gmat_hdf5
from keosop.numerical.keo import batch_Gmat, com
from keosop.revad import signed_plane_angle, distance, angle

jax.config.update("jax_enable_x64", True)

# converts G-matrix to cm^-1 units
G_to_invcm = (
    constants.value("Planck constant")
    * constants.value("Avogadro constant")
    * 1e16
    / (4.0 * np.pi**2 * constants.value("speed of light in vacuum"))
    * 1e5
)

Define the internal-to-Cartesian and Cartesian-to-internal coordinate transformations for $\mathrm{CH_3F}$.

The internal coordinates are defined as:

1. $r_{\mathrm{CF}}$ — C-F bond length,
2. $r_{\mathrm{CH_1}}$ — C-H1 bond length,
3. $r_{\mathrm{CH_2}}$ — C-H2 bond length,
4. $r_{\mathrm{CH_3}}$ — C-H3 bond length,
5. $\beta_{\mathrm{H_1CF}}$ — H1-C-F bond angle,
6. $\beta_{\mathrm{H_2CF}}$ — H2-C-F bond angle,
7. $\beta_{\mathrm{H_3CF}}$ — H3-C-F bond angle,
8. $s_1=(2\tau_1-\tau_2-\tau_3)/\sqrt{6}$ - symmetrized combination of $\tau_1 = \mathrm{H_2CFH_3}$, $\tau_2=\mathrm{H_3CFH_1}$, $\tau_3=\mathrm{H_1CFH_2}$ dihedral angles,
9. $s_2=(\tau_2-\tau_3)/\sqrt{2}$.

The atom ordering in Cartesian coordinates is: $\mathrm{C},\ \mathrm{F},\ \mathrm{H_1},\ \mathrm{H_2},\ \mathrm{H_3}$

In [2]:
masses = jnp.array([12.0, 18.9984, 1.007825, 1.007825, 1.007825])  # masses C, F, H, ...


def internal_to_cartesian(coords):
    r0, r1, r2, r3, beta1, beta2, beta3, stau1, stau2 = coords

    dbeta1 = np.pi - beta1
    dbeta2 = np.pi - beta2
    dbeta3 = np.pi - beta3

    tau1 = np.sqrt(6) / 3 * stau1 + 2 * np.pi / 3
    tau2 = -1 / np.sqrt(6) * stau1 + 1 / np.sqrt(2) * stau2 + 2 * np.pi / 3
    tau3 = -1 / np.sqrt(6) * stau1 - 1 / np.sqrt(2) * stau2 + 2 * np.pi / 3

    xyz = np.array(
        [
            [0.0, 0.0, 0.0],
            [0.0, 0.0, r0],
            [r1 * np.sin(dbeta1), 0.0, -r1 * np.cos(dbeta1)],
            [
                r2 * np.sin(dbeta2) * np.cos(tau3),
                r2 * np.sin(dbeta2) * np.sin(tau3),
                -r2 * np.cos(dbeta2),
            ],
            [
                r3 * np.sin(dbeta3) * np.cos(tau2),
                -r3 * np.sin(dbeta3) * np.sin(tau2),
                -r3 * np.cos(dbeta3),
            ],
        ],
        dtype=object,
    )

    return xyz


def cartesian_to_internal(xyz):
    C, F, H1, H2, H3 = xyz

    r0 = distance(C, F)
    r1 = distance(C, H1)
    r2 = distance(C, H2)
    r3 = distance(C, H3)

    beta1 = angle(F, C, H1)
    beta2 = angle(F, C, H2)
    beta3 = angle(F, C, H3)

    phi2 = signed_plane_angle((H1, C, F), (H2, C, F), (C, F))
    phi3 = signed_plane_angle((H1, C, F), (H3, C, F), (C, F))

    tau3 = phi2
    tau2 = -phi3
    tau1 = 2.0 * np.pi - tau2 - tau3

    stau1 = np.sqrt(6.0) / 2.0 * (tau1 - 2.0 * np.pi / 3.0)
    stau2 = (tau2 - tau3) / np.sqrt(2.0)

    return np.array([r0, r1, r2, r3, beta1, beta2, beta3, stau1, stau2], dtype=object)

Define one-dimensional grids for the internal coordinate representation and construct the G-matrix in sum-of-products form

In [3]:
rad = np.pi / 180

# define one-dimensional grids for each coordinate
grids = [
    np.linspace(0.8, 2.6, 3),  # r_{CF}
    np.linspace(0.8, 2.6, 3),  # r_{CH1}
    np.linspace(0.8, 2.6, 3),  # r_{CH2}
    np.linspace(0.8, 2.6, 3),  # r_{CH3}
    np.linspace(80 * rad, 170 * rad, 3),  # beta_{H1CF}
    np.linspace(80 * rad, 170 * rad, 3),  # beta_{H2CF}
    np.linspace(80 * rad, 170 * rad, 3),  # beta_{H3CF}
    np.linspace(-60 * rad, 60 * rad, 5),  # stau1
    np.linspace(-60 * rad, 60 * rad, 5),  # stau2
]

ncoo = len(grids)

# compute SOP for G-matrix
gmat = Gmat(grids, masses, internal_to_cartesian, cartesian_to_internal)

gvib_sop, grovib_sop, grot_sop = gmat.run(return_rot=True)

print("Number of terms for vibrational elements of G-matrix\n i  j no.terms")
for i in range(ncoo):
    for j in range(i, ncoo):
        print(f"{i:2d} {j:2d} {gvib_sop[i, j].nterms:6d}")

print("\nNumber of terms for rovibrational elements of G-matrix\n i  j no.terms")
for i in range(3):
    for j in range(ncoo):
        print(f"{i:2d} {j:2d} {grovib_sop[i, j].nterms:6d}")

print("\nNumber of terms for rotational elements of G-matrix\n i  j no.terms")
for i in range(3):
    for j in range(i, 3):
        print(f"{i:2d} {j:2d} {grot_sop[i, j].nterms:6d}")

Number of terms for vibrational elements of G-matrix
 i  j no.terms
 0  0      1
 0  1      1
 0  2      1
 0  3      1
 0  4      1
 0  5      1
 0  6      1
 0  7      1
 0  8      1
 1  1      1
 1  2      5
 1  3      5
 1  4      1
 1  5      9
 1  6      9
 1  7     16
 1  8     16
 2  2      1
 2  3      7
 2  4      9
 2  5      1
 2  6     13
 2  7     12
 2  8     20
 3  3      1
 3  4      9
 3  5     13
 3  6      1
 3  7     12
 3  8     20
 4  4      5
 4  5     17
 4  6     17
 4  7     32
 4  8     32
 5  5      5
 5  6     25
 5  7     24
 5  8     40
 6  6      5
 6  7     24
 6  8     40
 7  7     30
 7  8     38
 8  8     65

Number of terms for rovibrational elements of G-matrix
 i  j no.terms
 0  0      1
 0  1      1
 0  2      4
 0  3      4
 0  4      1
 0  5      8
 0  6      8
 0  7     16
 0  8     18
 1  0      1
 1  1      1
 1  2      4
 1  3      4
 1  4      2
 1  5      8
 1  6      8
 1  7     16
 1  8     16
 2  0      1
 2  1      1
 2  2      8
 2 

Evaluate the G-matrix numerically on the multidimensional grid formed by the Cartesian product of the one-dimensional grids used to construct the sum-of-products representation

In [4]:
@com(masses)
def internal_to_cartesian_jax(coords):
    r0, r1, r2, r3, beta1, beta2, beta3, stau1, stau2 = coords

    dbeta1 = jnp.pi - beta1
    dbeta2 = jnp.pi - beta2
    dbeta3 = jnp.pi - beta3

    tau1 = jnp.sqrt(6) / 3 * stau1 + 2 * jnp.pi / 3
    tau2 = -1 / jnp.sqrt(6) * stau1 + 1 / jnp.sqrt(2) * stau2 + 2 * jnp.pi / 3
    tau3 = -1 / jnp.sqrt(6) * stau1 - 1 / jnp.sqrt(2) * stau2 + 2 * jnp.pi / 3

    xyz = jnp.array(
        [
            [0.0, 0.0, 0.0],
            [0.0, 0.0, r0],
            [r1 * jnp.sin(dbeta1), 0.0, -r1 * jnp.cos(dbeta1)],
            [
                r2 * jnp.sin(dbeta2) * jnp.cos(tau3),
                r2 * jnp.sin(dbeta2) * jnp.sin(tau3),
                -r2 * jnp.cos(dbeta2),
            ],
            [
                r3 * jnp.sin(dbeta3) * jnp.cos(tau2),
                -r3 * jnp.sin(dbeta3) * jnp.sin(tau2),
                -r3 * jnp.cos(dbeta3),
            ],
        ]
    )
    return xyz


# build direct-product grid
product_grid = np.stack(np.meshgrid(*grids, indexing="ij"), axis=-1)
product_grid = product_grid.reshape(-1, len(grids))

# map each multidimensional grid point to its 1D grid indices
grid_indices = np.indices([len(grid) for grid in grids]).reshape(len(grids), -1).T

# compute G-matrix on grid
gmat = batch_Gmat(product_grid, masses, internal_to_cartesian_jax)

gvib_num = gmat[:, :ncoo, :ncoo]
grovib_num = gmat[:, ncoo : ncoo + 3, :ncoo]
grot_num = gmat[:, ncoo : ncoo + 3, ncoo : ncoo + 3]

Calculate point-wise differences between the sum-of-products and numerical $G$-matrix

In [5]:
# evaluate one SOP term
def eval_sop_term(sop, iterm, inds):
    value = 1.0
    for k in range(sop.ncoo):
        value *= sop.terms[iterm][k][inds[k]]
    return value

# evaluate a complete SumProd object at one direct-product grid point
def eval_sop(sop, inds):
    return sum(
        eval_sop_term(sop, iterm, inds)
        for iterm in range(sop.nterms)
    )

In [6]:
# number of multidimensional grid points
npoints = product_grid.shape[0]

gvib_sop_eval = np.zeros((npoints, ncoo, ncoo), dtype=np.float64)
grot_sop_eval = np.zeros((npoints, 3, 3), dtype=np.float64)
grovib_sop_eval = np.zeros((npoints, 3, ncoo), dtype=np.float64)

# Gvib
for ipoint, inds in enumerate(grid_indices):
    for i in range(ncoo):
        for j in range(ncoo):
            gvib_sop_eval[ipoint, i, j] = eval_sop(gvib_sop[i, j], inds) * G_to_invcm


# Grot
for ipoint, inds in enumerate(grid_indices):
    for i in range(3):
        for j in range(3):
            grot_sop_eval[ipoint, i, j] = eval_sop(grot_sop[i, j], inds) * G_to_invcm


# Grovib
for ipoint, inds in enumerate(grid_indices):
    for i in range(3):
        for j in range(ncoo):
            grovib_sop_eval[ipoint, i, j] = (
                eval_sop(grovib_sop[i, j], inds) * G_to_invcm
            )

In [7]:
# differences

diff_vib = gvib_sop_eval - gvib_num
diff_rovib = grovib_sop_eval - grovib_num
diff_rot = grot_sop_eval - grot_num

abs_diff_vib = np.abs(diff_vib)
abs_diff_rovib = np.abs(diff_rovib)
abs_diff_rot = np.abs(diff_rot)

print("--- Gvib ---")
print("Maximum absolute difference:", np.max(abs_diff_vib))
print("Mean absolute difference:", np.mean(abs_diff_vib))
print("RMS difference:", np.sqrt(np.mean(diff_vib**2)))

print("\n--- Grovib ---")
print("Maximum absolute difference:", np.max(abs_diff_rovib))
print("Mean absolute difference:", np.mean(abs_diff_rovib))
print("RMS difference:", np.sqrt(np.mean(diff_rovib**2)))

print("\n--- Grot ---")
print("Maximum absolute difference:", np.max(abs_diff_rot))
print("Mean absolute difference:", np.mean(abs_diff_rot))
print("RMS difference:", np.sqrt(np.mean(diff_rot**2)))

--- Gvib ---
Maximum absolute difference: 3.2896423363126814e-09
Mean absolute difference: 8.609820540772733e-13
RMS difference: 1.9179298840765985e-11

--- Grovib ---
Maximum absolute difference: 2.325123205082491e-09
Mean absolute difference: 1.128199123778031e-12
RMS difference: 2.1685205812573404e-11

--- Grot ---
Maximum absolute difference: 1.6457306628581136e-09
Mean absolute difference: 1.9951589630453426e-12
RMS difference: 2.5401874044931705e-11


In [8]:
print("--- Gvib element-wise errors ---")

for i in range(ncoo):
    for j in range(i, ncoo):
        d = diff_vib[:, i, j]
        max_abs = np.max(np.abs(d))
        rms = np.sqrt(np.mean(d**2))
        print(f"Gvib[{i},{j}]  " f"max abs = {max_abs:.3e}   " f"RMS = {rms:.3e}")

print("\n--- Grovib element-wise errors ---")

for i in range(3):
    for j in range(ncoo):
        d = diff_rovib[:, i, j]
        max_abs = np.max(np.abs(d))
        rms = np.sqrt(np.mean(d**2))
        print(f"Grovib[{i},{j}]  " f"max abs = {max_abs:.3e}   " f"RMS = {rms:.3e}")

print("\n--- Grot element-wise errors ---")

for i in range(3):
    for j in range(i, 3):
        d = diff_rot[:, i, j]
        max_abs = np.max(np.abs(d))
        rms = np.sqrt(np.mean(d**2))
        print(f"Grot[{i},{j}]  " f"max abs = {max_abs:.3e}   " f"RMS = {rms:.3e}")

--- Gvib element-wise errors ---
Gvib[0,0]  max abs = 2.665e-15   RMS = 1.081e-15
Gvib[0,1]  max abs = 2.887e-15   RMS = 5.593e-16
Gvib[0,2]  max abs = 3.997e-15   RMS = 7.945e-16
Gvib[0,3]  max abs = 3.331e-15   RMS = 7.578e-16
Gvib[0,4]  max abs = 6.661e-15   RMS = 6.941e-16
Gvib[0,5]  max abs = 7.105e-15   RMS = 6.965e-16
Gvib[0,6]  max abs = 9.215e-15   RMS = 6.801e-16
Gvib[0,7]  max abs = 3.327e-13   RMS = 8.317e-15
Gvib[0,8]  max abs = 4.217e-13   RMS = 1.329e-14
Gvib[1,1]  max abs = 4.974e-14   RMS = 1.038e-14
Gvib[1,2]  max abs = 3.997e-14   RMS = 2.662e-15
Gvib[1,3]  max abs = 3.741e-14   RMS = 2.608e-15
Gvib[1,4]  max abs = 7.505e-14   RMS = 5.921e-15
Gvib[1,5]  max abs = 6.839e-14   RMS = 4.560e-15
Gvib[1,6]  max abs = 6.128e-14   RMS = 4.443e-15
Gvib[1,7]  max abs = 3.645e-12   RMS = 1.894e-13
Gvib[1,8]  max abs = 2.585e-12   RMS = 1.207e-13
Gvib[2,2]  max abs = 3.837e-13   RMS = 1.588e-14
Gvib[2,3]  max abs = 3.220e-13   RMS = 9.379e-15
Gvib[2,4]  max abs = 3.118e-13   RMS

In [9]:
print("--- Gvib largest discrepancy ---")
idx = np.unravel_index(np.argmax(abs_diff_vib), abs_diff_vib.shape)
ipoint, i, j = idx
print("grid point :", ipoint)
print("coordinates:", np.asarray(product_grid[ipoint]))
print("element    :", (i, j))
print("SOP        :", gvib_sop_eval[ipoint, i, j])
print("reference  :", gvib_num[ipoint, i, j])
print("difference :", diff_vib[ipoint, i, j])

print("\n--- Grovib largest discrepancy ---")
idx = np.unravel_index(np.argmax(abs_diff_rovib), abs_diff_rovib.shape)
ipoint, i, j = idx
print("grid point :", ipoint)
print("coordinates:", np.asarray(product_grid[ipoint]))
print("element    :", (i, j))
print("SOP        :", grovib_sop_eval[ipoint, i, j])
print("reference  :", grovib_num[ipoint, i, j])
print("difference :", diff_rovib[ipoint, i, j])

print("\n--- Grot largest discrepancy ---")
idx = np.unravel_index(np.argmax(abs_diff_rot), abs_diff_rot.shape)
ipoint, i, j = idx
print("grid point :", ipoint)
print("coordinates:", np.asarray(product_grid[ipoint]))
print("element    :", (i, j))
print("SOP        :", grot_sop_eval[ipoint, i, j])
print("reference  :", grot_num[ipoint, i, j])
print("difference :", diff_rot[ipoint, i, j])

--- Gvib largest discrepancy ---
grid point : 5851
coordinates: [ 0.8         0.8         2.6         2.6         2.96705973  1.3962634
  1.3962634  -1.04719755 -0.52359878]
element    : (np.int64(8), np.int64(8))
SOP        : 4810.06151051063
reference  : 4810.06151050734
difference : 3.2896423363126814e-09

--- Grovib largest discrepancy ---
grid point : 5851
coordinates: [ 0.8         0.8         2.6         2.6         2.96705973  1.3962634
  1.3962634  -1.04719755 -0.52359878]
element    : (np.int64(2), np.int64(8))
SOP        : 3393.1078694834464
reference  : 3393.1078694811213
difference : 2.325123205082491e-09

--- Grot largest discrepancy ---
grid point : 5851
coordinates: [ 0.8         0.8         2.6         2.6         2.96705973  1.3962634
  1.3962634  -1.04719755 -0.52359878]
element    : (np.int64(2), np.int64(2))
SOP        : 2396.210744555456
reference  : 2396.2107445538104
difference : 1.6457306628581136e-09


Now, generate the sum-of-products KEO using larger 1D grids and store the results in an HDF5 file

In [10]:
rad = np.pi / 180

# define one-dimensional grids for each coordinate
grids = [
    np.linspace(0.8, 2.6, 1000),  # r_{CF}
    np.linspace(0.8, 2.6, 1000),  # r_{CH1}
    np.linspace(0.8, 2.6, 1000),  # r_{CH2}
    np.linspace(0.8, 2.6, 1000),  # r_{CH3}
    np.linspace(80 * rad, 179 * rad, 1000),  # beta_{H1CF}
    np.linspace(80 * rad, 179 * rad, 1000),  # beta_{H2CF}
    np.linspace(80 * rad, 179 * rad, 1000),  # beta_{H3CF}
    np.linspace(-60 * rad, 60 * rad, 1000),  # stau1
    np.linspace(-60 * rad, 60 * rad, 1000),  # stau2
]

ncoo = len(grids)

# compute SOP for G-matrix
gmat = Gmat(grids, masses, internal_to_cartesian, cartesian_to_internal)

gvib_sop, grovib_sop, grot_sop = gmat.run(return_rot=True)

# store into file
filename = "keo_ch3f.h5"
save_gmat_hdf5(filename, grids, gvib_sop, grovib_sop, grot_sop)